# Double-forms



In [ ]:
from ngsolve import *
from netgen.occ import unit_square, unit_cube
import ngsdiffgeo as dg
import math


TOL = 1e-8


def l2_error(a, b, mesh):
    return sqrt(Integrate(InnerProduct(a - b, a - b), mesh))


def l2_norm(a, mesh):
    return sqrt(Integrate(InnerProduct(a, a), mesh))

In [ ]:
mesh3 = Mesh(unit_cube.GenerateMesh(maxh=2))

dx = dg.OneForm(CF((1, 0, 0)))
dy = dg.OneForm(CF((0, 1, 0)))
dz = dg.OneForm(CF((0, 0, 1)))

f = dg.ScalarField(20 * x * y * (1 - y) * (1 - z), dim=3)

alpha1 = dg.OneForm(CF((0.3 * x * y, z**2, -0.1 * x)))
beta1 = dg.OneForm(CF((0.3 * z * y, x * z**2, y**2)))
gamma1 = dg.OneForm(CF((0.3 * y * z - x * y, x**2 * z + 0.34 * y**3, -x * y * z)))

alpha2 = dg.TwoForm(
    CF(
        (
            0,
            -0.1 * z**2 - x * y,
            x * y,
            -(-0.1 * z**2 - x * y),
            0,
            0.23 * x * y * z,
            -x * y,
            -0.23 * x * y * z,
            0,
        ),
        dims=(3, 3),
    )
)
beta2 = dg.TwoForm(
    CF(
        (
            0,
            y * z - 0.1 * x**2,
            0,
            -(y * z - 0.1 * x**2),
            0,
            x * z,
            0,
            -x * z,
            0,
        ),
        dims=(3, 3),
    )
)

A11 = dg.DoubleForm(dg.Einsum("i,j->ij", dx, dy), p=1, q=1, dim=3)
B11 = dg.DoubleForm(dg.Einsum("i,j->ij", alpha1, beta1), p=1, q=1, dim=3)
C11 = dg.DoubleForm(dg.Einsum("i,j->ij", beta1, gamma1), p=1, q=1, dim=3)

A12 = dg.DoubleForm(dg.Einsum("i,jk->ijk", gamma1, alpha2), p=1, q=2, dim=3)
B12 = dg.DoubleForm(dg.Einsum("i,jk->ijk", beta1, beta2), p=1, q=2, dim=3)
C21 = dg.DoubleForm(dg.Einsum("ij,k->ijk", alpha2, dx), p=2, q=1, dim=3)

A22 = dg.Wedge(B11, C11)
B22 = dg.DoubleForm(dg.Einsum("ij,kl->ijkl", alpha2, beta2), p=2, q=2, dim=3)

A33 = dg.Wedge(A22, B11)
B33 = dg.Wedge(B22, C11)


Define with $g^{-1}(\cdot,\cdot)$ the canonical inner product of $k$-forms and double-forms. Further denote by $\langle\cdot,\cdot\rangle$ the canonical $g$ inner product of tensor fields. There holds the relation

$$
\begin{align*}
g^{-1}(\varphi,\Psi)=\frac{1}{p!q!}\langle \varphi,\Psi\rangle,\qquad \varphi,\Psi\in \Lambda^{p,q}(\Omega).
\end{align*}
$$

In [ ]:
# mf = dg.RiemannianManifold(Id(3))
mf = dg.RiemannianManifold(dg.WarpedProduct().metric)

left = mf.InnerProduct(B11, C11)
right = mf.InnerProduct(B11, C11, forms=True)

print(f"L2 error inner product (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = (
    1
    / (math.factorial(A12.degree_left) * math.factorial(A12.degree_right))
    * mf.InnerProduct(A12, B12)
)
right = mf.InnerProduct(A12, B12, forms=True)
print(f"L2 error inner product (1,2)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

The trace of a double-form is defined for an orthonormal basis $\{e_i\}_{i=1}^N$ by
$$
\begin{align*}
\mathrm{Tr}:\Lambda^{p,q}(\Omega)\to\Lambda^{p-1,q-1}:\quad \mathrm{Tr}(\alpha\otimes\beta)=\sum_{i=1}^N(\iota_{e_i}\alpha)\otimes(\iota_{e_i}\beta),
\end{align*}
$$
where $\iota_X$ is the contraction operator for $k$-forms. I.e., the trace contracts the first slot of the left with the first slot of the right part of the double-form.

We also define the mapping
$g^{-1}(\cdot):\Lambda^{p,p}(\Omega)\to\mathbb{R}$ for $\varphi=\alpha\otimes\beta$, $\alpha,\beta\in\Lambda^p(\Omega)$ by 
$$
\begin{align*}
g^{-1}(A)=g^{-1}(\alpha,\beta).
\end{align*}
$$
There holds for $\mathrm{Tr}^l:= \mathrm{Tr}\circ\cdots\circ\mathrm{Tr}$
$$
\begin{align*}
g^{-1}(\varphi) = \frac{1}{p!}\mathrm{Tr}^p\varphi,\qquad\varphi\in\Lambda^{p,p}(\Omega).
\end{align*}
$$

In [ ]:
left = dg.slot_inner_product(A11, mf)
right = mf.Trace(A11)
print(f"L2 error trace (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = dg.slot_inner_product(B22, mf, forms=True)
right = 1 / (math.factorial(B22.degree_left)) * mf.Trace(B22, l=2)
print(f"L2 error trace (2,2)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

Analogously to $k$-forms we can define a wedge operator and a Hodge star operator for double-forms, defined as 
$$
\begin{align*}
&\wedge:\Lambda^{k,l}(\Omega)\times \Lambda^{p,q}(\Omega)\to\Lambda^{k+p,l+q}(\Omega):\quad \wedge(\alpha\otimes\beta)\wedge (\gamma\otimes\gamma) = (\alpha\wedge\gamma)\otimes (\beta\wedge\delta),\\
&\star:\Lambda^{p,q}(\Omega)\to\Lambda^{N-p,N-q}(\Omega):\quad \star(\alpha\otimes\beta) = \star\alpha\otimes\star\beta.
\end{align*}
$$

For example, there holds the relation
$$
\begin{align*}
g^{-1}(\varphi,\Psi) = \star^{-1}(\varphi\wedge\star\Psi),\qquad \varphi,\Psi\in\Lambda^{p,q}(\Omega).
\end{align*}
$$

In [ ]:
left = mf.InnerProduct(C11, B11, forms=True)
right = mf.inv_star(dg.Wedge(C11, mf.star(B11)))
print(f"L2 (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = mf.InnerProduct(A12, B12, forms=True)
right = mf.inv_star(dg.Wedge(A12, mf.star(B12)))
print(f"L2 (1,2)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL


left = mf.InnerProduct(A22, B22, forms=True)
right = mf.inv_star(dg.Wedge(A22, mf.star(B22)))
print(f"L2 (2,2)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

There holds for $\varphi\in \Lambda^{p,q}(\Omega)$ and $\Psi\in \Lambda^{p+1,q+1}(\Omega)$
$$
\begin{align*}
g^{-1}(g\wedge\varphi,\Psi)= g^{-1}(\varphi,\mathrm{Tr}(\Psi)),
\end{align*}
$$
i.e., $g\wedge$ is adjoint to $\mathrm{Tr}$.

In [ ]:
left = mf.InnerProduct(dg.Wedge(mf.G, B11), B22, forms=True)
right = mf.InnerProduct(B11, mf.Trace(B22), forms=True)
print(
    f"L2 error wedge-trace dual (1,1)-(2,2)-doubleforms: {l2_error(left, right, mesh3):.6f}"
)
assert l2_error(left, right, mesh3) < TOL

A23 = dg.Wedge(A12, B11)
left = mf.InnerProduct(dg.Wedge(mf.G, B12), A23, forms=True)
right = mf.InnerProduct(B12, mf.Trace(A23), forms=True)
print(
    f"L2 error wedge-trace dual (1,2)-(2,3)-doubleforms: {l2_error(left, right, mesh3):.6f}"
)
assert l2_error(left, right, mesh3) < TOL


Let $g^l=g\wedge\cdots\wedge g$. There holds for all $\varphi\in \Lambda^{p,q}(\Omega)$ and $0\le l\le\min\{N-p,N-q\}$
$$
\begin{align*}
\mathrm{Tr}^l\star\varphi = \star(\varphi\wedge g^l).
\end{align*}
$$

In [ ]:
G2 = dg.Wedge(mf.G, mf.G)
G3 = dg.Wedge(G2, mf.G)

left = mf.Trace(mf.star(B11), l=1)
right = mf.star(dg.Wedge(B11, mf.G))
print(f"L2 error (1,1)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = mf.Trace(mf.star(B11), l=2)
right = mf.star(dg.Wedge(B11, G2))
print(f"L2 error (1,1)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = mf.Trace(mf.star(B22), l=1)
right = mf.star(dg.Wedge(B22, mf.G))
print(f"L2 error (2,2)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = mf.Trace(mf.star(C21), l=1)
right = mf.star(dg.Wedge(C21, mf.G))
print(f"L2 error (2,1)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = mf.Trace(mf.star(B12), l=1)
right = mf.star(dg.Wedge(B12, mf.G))
print(f"L2 error (1,2)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

For all $\varphi\in \Lambda^{p,p}(\Omega)$ and $0\le l \le N-p$ there holds
$$
\begin{align*}
g^{-1}(\varphi\wedge g^l) = \frac{(N-p)!}{(N-p-l)!}g^{-1}(\varphi).
\end{align*}
$$

In [ ]:
N, p = 3, 1
left = dg.slot_inner_product(dg.Wedge(B11, mf.G), mf)
right = (
    math.factorial(N - p) / math.factorial(N - p - 1) * dg.slot_inner_product(B11, mf)
)
print(f"L2 error (1,1)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p = 3, 1
left = dg.slot_inner_product(dg.Wedge(B11, dg.Wedge(mf.G, mf.G)), mf)
right = (
    math.factorial(N - p) / math.factorial(N - p - 2) * dg.slot_inner_product(B11, mf)
)
print(f"L2 error (1,1)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL


N, p = 3, 2
left = dg.slot_inner_product(dg.Wedge(A22, mf.G), mf)
right = (
    math.factorial(N - p) / math.factorial(N - p - 1) * dg.slot_inner_product(A22, mf)
)
print(f"L2 error (2,2)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

For all $\varphi\in\Lambda^{p,p}(\Omega)$ there holds
$$
\begin{align*}
\mathrm{Tr}(\varphi\wedge g) = (N-2p)\varphi+(\mathrm{Tr}\varphi)\wedge g
\end{align*}
$$
and for all $l\ge 1$
$$
\begin{align*}
\mathrm{Tr}(\varphi\wedge g^l)=l(N-2p-l+1)\varphi\wedge g^{l-1}+\mathrm{Tr}\varphi\wedge g^l.
\end{align*}
$$

In [ ]:
N, p = 3, 1
left = mf.Trace(dg.Wedge(B11, mf.G))
right = (N - 2 * p) * B11 + dg.Wedge(mf.Trace(B11), mf.G)
print(f"L2 error (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 1, 2
G2 = dg.Wedge(mf.G, mf.G)
left = mf.Trace(dg.Wedge(B11, G2))
right = l * (N - 2 * p - l + 1) * dg.Wedge(B11, mf.G) + dg.Wedge(mf.Trace(B11), G2)
print(f"L2 error (1,1)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p = 3, 2
left = mf.Trace(dg.Wedge(A22, mf.G))
right = (N - 2 * p) * A22 + dg.Wedge(mf.Trace(A22), mf.G)
print(f"L2 error (2,2)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 2, 2
G2 = dg.Wedge(mf.G, mf.G)
left = mf.Trace(dg.Wedge(A22, G2))
right = l * (N - 2 * p - l + 1) * dg.Wedge(A22, mf.G) + dg.Wedge(mf.Trace(A22), G2)
print(f"L2 error (2,2)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

For all $\varphi\in\Lambda^{p,p}(\Omega)$ and all $l\ge 1$ there holds
$$
\begin{align*}
\mathrm{Tr}^l(\varphi\wedge g) = l(N-2p+l-1)\mathrm{Tr}^{l-1}\varphi+\mathrm{Tr}\varphi^l\wedge g.
\end{align*}
$$

In [ ]:
N, p, l = 3, 1, 1
left = mf.Trace(dg.Wedge(B11, mf.G), l=l)
right = l * (N - 2 * p + l - 1) * mf.Trace(B11, l=l - 1) + dg.Wedge(mf.Trace(B11), mf.G)
print(f"L2 error (1,1)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 1, 2
left = mf.Trace(dg.Wedge(B11, mf.G), l=l)
right = l * (N - 2 * p + l - 1) * mf.Trace(B11, l=l - 1) + dg.Wedge(
    mf.Trace(B11, l=l), mf.G
)
print(f"L2 error (1,1)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 2, 1
left = mf.Trace(dg.Wedge(A22, mf.G), l=l)
right = l * (N - 2 * p + l - 1) * mf.Trace(A22, l=l - 1) + dg.Wedge(mf.Trace(A22), mf.G)
print(f"L2 error (2,2)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 2, 2
left = mf.Trace(dg.Wedge(A22, mf.G), l=l)
right = l * (N - 2 * p + l - 1) * mf.Trace(A22, l=l - 1) + dg.Wedge(
    mf.Trace(A22, l=l), mf.G
)
print(f"L2 error (2,2)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

For all $p\ge l\ge0$ there holds
$$
\begin{align*}
\frac{1}{l!}\mathrm{Tr}^l\left(\frac{1}{p!}g^p\right) = \binom{N-p+l}{l}\frac{1}{(p-l)!}g^{p-l}
\end{align*}
$$

In [ ]:
g = [
    dg.ScalarField(1, dim=3),
    dg.DoubleForm(mf.G, p=1, q=1, dim=3),
    dg.Wedge(mf.G, mf.G),
    dg.Wedge(dg.Wedge(mf.G, mf.G), mf.G),
]


def term_left(p, l):
    if l < 0 or p < l:
        raise ValueError("Invalid values for p and l.")
    return 1 / math.factorial(l) * mf.Trace(1 / math.factorial(p) * g[p], l=l)


def term_right(p, l):
    if l < 0 or p < l:
        raise ValueError("Invalid values for p and l.")
    return math.comb(N - p + l, l) * 1 / math.factorial(p - l) * g[p - l]


for p in range(3):
    for l in range(p + 1):
        left = term_left(p, l)
        right = term_right(p, l)
        print(f"L2 error p={p}, l={l}: {l2_error(left, right, mesh3):.6f}")
        assert l2_error(left, right, mesh3) < TOL

There holds with the volume form $\omega\in\Lambda^N(\Omega)$
$$
\begin{align*}
\frac{1}{N!}g^N = \omega\otimes\omega.
\end{align*}
$$

In [ ]:
omega = mf.VolumeForm(VOL) * dg.Wedge(dx, dg.Wedge(dy, dz))

left = 1 / math.factorial(3) * dg.Wedge(dg.Wedge(mf.G, mf.G), mf.G)
right = dg.TensorProduct(omega, omega)
print(f"L2 error volume form: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

There holds
$$
\begin{align*}
\frac{1}{(N-1)!}\mathrm{Tr}^{N-1}(\omega\otimes\omega) = g.
\end{align*}
$$

In [ ]:
omega = mf.VolumeForm(VOL) * dg.Wedge(dx, dg.Wedge(dy, dz))
N = 3

left = (
    1
    / math.factorial(N - 1)
    * mf.Trace(dg.DoubleForm(dg.TensorProduct(omega, omega), p=3, q=3, dim=3), l=N - 1)
)
right = mf.G
print(f"L2 error metric from volume form: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

There holds for all $\varphi,\Psi\in\Lambda^{p,p}(\Omega)$
$$
\begin{align*}
g^{-1}(\varphi\wedge g,\Psi\wedge g) = (N-2p)g^{-1}(\varphi,\Psi) + g^{-1}(\mathrm{Tr}\varphi,\mathrm{Tr}\Psi)
\end{align*}
$$

In [ ]:
N = 3


def left_term(phi, psi):
    return mf.InnerProduct(dg.Wedge(phi, mf.G), dg.Wedge(psi, mf.G), forms=True)


def right_term(phi, psi):
    p = phi.degree_left
    return (N - 2 * p) * mf.InnerProduct(phi, psi, forms=True) + mf.InnerProduct(
        mf.Trace(phi), mf.Trace(psi), forms=True
    )


left = left_term(B11, C11)
right = right_term(B11, C11)
print(f"L2 error (1,1)-doubleforms via formula: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = left_term(A22, B22)
right = right_term(A22, B22)
print(f"L2 error (2,2)-doubleforms via formula: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL


left = left_term(A33, B33)
right = right_term(A33, B33)
print(f"L2 error (3,3)-doubleforms via formula: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL


For all $\varphi\in\Lambda^{p,p}$ there holds
$$
\begin{align*}
\mathrm{Tr}\left( \sum_{k=0}^{p}(-1)^{p-k}\frac{(N-2p+1)!}{(N-p-k+1)!(p-k)!}\mathrm{Tr}^{p-k}\varphi\wedge g^{p-k} \right)=0
\end{align*}
$$

In [ ]:
G = mf.G

gpk = [
    [G, dg.ScalarField(CF(1), dim=3)],
    [dg.Wedge(G, G), G, dg.ScalarField(CF(1), dim=3)],
    [dg.Wedge(G, dg.Wedge(G, G)), dg.Wedge(G, G), G, dg.ScalarField(CF(1), dim=3)],
]

N, p = 3, 1
dev_B11 = sum(
    (-1) ** (p - k)
    * math.factorial(N - 2 * p + 1)
    / (math.factorial(N - p - k + 1) * math.factorial(p - k))
    * dg.Wedge(mf.Trace(B11, l=p - k), gpk[p - 1][k])
    for k in range(0, p + 1)
)
should_be_zero = mf.Trace(dev_B11)
print(
    f"L2 norm of should be zero (1,1)-doubleforms: {l2_norm(should_be_zero, mesh3):.6f}"
)
assert l2_norm(should_be_zero, mesh3) < TOL

N, p = 3, 2
dev_A22 = sum(
    (-1) ** (p - k)
    * math.factorial(N - 2 * p + 1)
    / (math.factorial(N - p - k + 1) * math.factorial(p - k))
    * dg.Wedge(mf.Trace(A22, l=p - k), gpk[p - 1][k])
    for k in range(0, p + 1)
)
should_be_zero = mf.Trace(dev_A22)
print(
    f"L2 norm of should be zero (2,2)-doubleforms: {l2_norm(should_be_zero, mesh3):.6f}"
)
assert l2_norm(should_be_zero, mesh3) < TOL

N, p = 3, 3
# dev_A33 = sum(
#     (-1) ** (p - k)
#     * math.factorial(N - 2 * p + 1)
#     / (math.factorial(N - p - k + 1) * math.factorial(p - k))
#     * dg.Wedge(mf.Trace(A33, l=p - k), gpk[p - 1][k])
#     for k in range(0, p + 1)
# )
# should_be_zero = mf.Trace(dev_A33)
# print(
#     f"L2 norm of should be zero (3,3)-doubleforms: {l2_norm(should_be_zero, mesh3):.6f}"
# )
# assert l2_norm(should_be_zero, mesh3) < TOL

Let $\varphi,\Psi\in \Lambda^{p,p}(\Omega)$. If $\mathrm{Tr}\varphi=0$ then
$$
\begin{align*}
g^{-1}(\varphi^T,\Psi)= (-1)^pg^{-1}(\varphi\wedge\Psi).
\end{align*}
$$

In [ ]:
left = mf.InnerProduct(dev_B11.trans, C11, forms=True)
right = (-1) ** 1 * mf.SlotInnerProduct(dg.Wedge(dev_B11, C11))
print(
    f"L2 error doubleform identity (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}"
)
assert l2_error(left, right, mesh3) < TOL

left = mf.InnerProduct(dev_A22.trans, B22, forms=True)
right = (-1) ** 2 * mf.SlotInnerProduct(dg.Wedge(dev_A22, B22))
print(
    f"L2 error doubleform identity (2,2)-doubleforms: {l2_error(left, right, mesh3):.6f}"
)
assert l2_error(left, right, mesh3) < TOL

Let $\varphi\in\Lambda^{l,l}(\Omega)$ and $\Psi\in\Lambda^{p,p}(\Omega)$, where $l <p$. If $\mathrm{Tr}\Psi=0$, then
$$
\begin{align*}
g^{-1}(\varphi\wedge \Psi)=0.
\end{align*}
$$

In [ ]:
should_be_zero = mf.SlotInnerProduct(dg.Wedge(B11, dev_A22))
print(f"L2 norm of should be zero (1,1)-(2,2) {l2_norm(should_be_zero, mesh3):.6f}")
assert l2_norm(should_be_zero, mesh3) < TOL

# TODO dev_A33 not available for N=3?
# should_be_zero = mf.SlotInnerProduct(dg.Wedge(B11, dev_A33))
# print(f"L2 norm of should be zero (1,1)-(3,3) {l2_norm(should_be_zero, mesh3):.6f}")
# assert l2_norm(should_be_zero, mesh3) < TOL

# TODO dev_A33 not available for N=3?
# should_be_zero = mf.SlotInnerProduct(dg.Wedge(A22, dev_A33))
# print(f"L2 norm of should be zero (2,2)-(3,3) {l2_norm(should_be_zero, mesh3):.6f}")
# assert l2_norm(should_be_zero, mesh3) < TOL

For all $\varphi,\Psi\in \Lambda^{p,p}(\Omega)$ there holds
$$
\begin{align*}
g^{-1}(\varphi,\Psi)=\sum_{k=0}^p\frac{(-1)^k}{(p-k)!}g^{-1}(\mathrm{Tr}^{p-k}\varphi^T\wedge \Psi).
\end{align*}
$$

In [ ]:
p = 1
left = mf.InnerProduct(A11, B11, forms=True)
right = sum(
    (-1) ** k
    / (math.factorial(p - k))
    * mf.SlotInnerProduct(dg.Wedge(mf.Trace(A11.trans, l=p - k), B11))
    for k in range(0, p + 1)
)
print(f"L2 error (1,1): {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

p = 2
left = mf.InnerProduct(A22, B22, forms=True)
right = sum(
    (-1) ** k
    / (math.factorial(p - k))
    * mf.SlotInnerProduct(dg.Wedge(mf.Trace(A22.trans, l=p - k), B22))
    for k in range(0, p + 1)
)
print(f"L2 error (2,2): {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

# TODO dg.Wedge(mf.Trace(A33.trans,l=0), B33) yields rank 12 tensorfield which
# TODO exceeds limitation of rank 8 in NGSDiffGeo
# p = 3
# left = mf.InnerProduct(A33, B33, forms=True)
# right = sum(
#     (-1) ** k
#     / (math.factorial(p - k))
#     * mf.SlotInnerProduct(dg.Wedge(mf.Trace(A33.trans,l=p-k), B33))
#     for k in range(0, p + 1)
# )
# print(f"L2 error (3,3): {l2_error(left, right, mesh3):.6f}")
# assert l2_error(left, right, mesh3) < TOL


Define
$$
\begin{align*}
B(\varphi,\Psi):=\sum_{k=0}^p\frac{(-1)^k}{(p-k)!}g^{-1}(\mathrm{Tr}^{p-k}\varphi^T\wedge \Psi).
\end{align*}
$$
Then there holds
$$
\begin{align*}
&B(\varphi,\Psi) = g^{-1}(\varphi,\Psi),\qquad \forall \varphi\in\Sigma_0^{p,p}(\Omega),\Psi\in\Lambda^{p,p}(\Omega),\\
&B(\varphi,\Psi) = g^{-1}(\varphi,\Psi),\qquad \forall \varphi\in\Lambda^{p,p}(\Omega),\Psi\perp \Sigma_0^{p,p}(\Omega),\\
&B(\varphi,\Psi) = 0 = g^{-1}(\varphi,\Psi),\qquad \forall \varphi \perp \Sigma_0^{p,p}(\Omega),\Psi\in\Sigma_0^{p,p}(\Omega).
\end{align*}
$$

In [ ]:
def B(phi, psi):
    p = phi.degree_left
    return sum(
        (-1) ** k
        / (math.factorial(p - k))
        * mf.SlotInnerProduct(dg.Wedge(mf.Trace(phi.trans, l=p - k), psi))
        for k in range(0, p + 1)
    )


def dev_dform(phi):
    p = phi.degree_left
    N = phi.dim_space
    gpk = [
        [G, dg.ScalarField(CF(1), dim=N)],
        [dg.Wedge(G, G), G, dg.ScalarField(CF(1), dim=N)],
        [dg.Wedge(G, dg.Wedge(G, G)), dg.Wedge(G, G), G, dg.ScalarField(CF(1), dim=N)],
    ]

    return sum(
        (-1) ** (p - k)
        * math.factorial(N - 2 * p + 1)
        / (math.factorial(N - p - k + 1) * math.factorial(p - k))
        * dg.Wedge(mf.Trace(phi, l=p - k), gpk[p - 1][k])
        for k in range(0, p + 1)
    )


tr_B11 = B11 - dev_B11
tr_A22 = A22 - dev_A22

should_be_zero = B(tr_B11, dev_dform(C11))
print(
    f"L2 norm of should be zero (1,1)-doubleforms: {l2_norm(should_be_zero, mesh3):.6f}"
)
assert l2_norm(should_be_zero, mesh3) < TOL

should_be_zero = B(tr_A22, dev_dform(B22))
print(
    f"L2 norm of should be zero (2,2)-doubleforms: {l2_norm(should_be_zero, mesh3):.6f}"
)
assert l2_norm(should_be_zero, mesh3) < TOL

For $\varphi\in\Lambda^{p,p}$ there holds for all $l=0,1,\dots N-p$
$$
\begin{align*}
\frac{1}{(N-p-l)!}\star\mathrm{Tr}^{N-p-l}\star\varphi = \sum_{k\ge 0}\frac{(-1)^k}{(l-k)!(p-k)!}\mathrm{Tr}^{l-k}\star\mathrm{Tr}^{p-k}\varphi^T.
\end{align*}
$$

In [ ]:
N, p, l = 3, 1, 0
left = (
    1
    / math.factorial(N - p - l)
    * mf.star(mf.Trace(mf.star(B11), l=N - p - l), double=True)
)
right = sum(
    (-1) ** k
    / (math.factorial(l - k) * math.factorial(p - k))
    * mf.Trace(mf.star(mf.Trace(B11.trans, l=p - k), double=True), l=l - k)
    for k in range(0, min(l, p) + 1)
)
print(f"L2 error (1,1)-doubleforms l=0: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 1, 1
left = (
    1
    / math.factorial(N - p - l)
    * mf.star(mf.Trace(mf.star(B11), l=N - p - l), double=True)
)
right = sum(
    (-1) ** k
    / (math.factorial(l - k) * math.factorial(p - k))
    * mf.Trace(mf.star(mf.Trace(B11.trans, l=p - k), double=True), l=l - k)
    for k in range(0, min(l, p) + 1)
)
print(f"L2 error (1,1)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 1, 2
left = (
    1
    / math.factorial(N - p - l)
    * mf.star(mf.Trace(mf.star(B11), l=N - p - l), double=True)
)
right = sum(
    (-1) ** k
    / (math.factorial(l - k) * math.factorial(p - k))
    * mf.Trace(mf.star(mf.Trace(B11.trans, l=p - k), double=True), l=l - k)
    for k in range(0, min(l, p) + 1)
)
print(f"L2 error (1,1)-doubleforms l=2: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 2, 0
left = (
    1
    / math.factorial(N - p - l)
    * mf.star(mf.Trace(mf.star(A22), l=N - p - l), double=True)
)
right = sum(
    (-1) ** k
    / (math.factorial(l - k) * math.factorial(p - k))
    * mf.Trace(mf.star(mf.Trace(A22.trans, l=p - k), double=True), l=l - k)
    for k in range(0, min(l, p) + 1)
)
print(f"L2 error (2,2)-doubleforms l=0: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 2, 1
left = (
    1
    / math.factorial(N - p - l)
    * mf.star(mf.Trace(mf.star(A22), l=N - p - l), double=True)
)
right = sum(
    (-1) ** k
    / (math.factorial(l - k) * math.factorial(p - k))
    * mf.Trace(mf.star(mf.Trace(A22.trans, l=p - k), double=True), l=l - k)
    for k in range(0, min(l, p) + 1)
)
print(f"L2 error (2,2)-doubleforms l=1: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

N, p, l = 3, 3, 0
left = (
    1
    / math.factorial(N - p - l)
    * mf.star(mf.Trace(mf.star(A33), l=N - p - l), double=True)
)
right = sum(
    (-1) ** k
    / (math.factorial(l - k) * math.factorial(p - k))
    * mf.Trace(mf.star(mf.Trace(A33.trans, l=p - k), double=True), l=l - k)
    for k in range(0, min(l, p) + 1)
)
print(f"L2 error (3,3)-doubleforms l=0: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL


For all $\varphi\in\Lambda^{p,p}$ and $\Psi\in\Lambda^{q,q}(\Omega)$  with $p+q \le N$ there holds
$$
\begin{align*}
g^{-1}(\varphi\wedge\Psi)=\frac{1}{(N-p-q)!}g^{-1}(\mathrm{Tr}^{N-p-q}\star\varphi,\Psi) = \sum_{k\ge 0}\frac{(-1)^k}{(q-k)!(p-k)!}g^{-1}(\mathrm{Tr}^{p-k}\varphi^T,\mathrm{Tr}^{q-k}\Psi).
\end{align*}
$$

In [ ]:
def middle_term(phi, psi):
    p = phi.degree_left
    q = psi.degree_left
    N = phi.dim_space

    return (
        1
        / math.factorial(N - p - q)
        * mf.InnerProduct(mf.Trace(mf.star(phi), l=N - p - q), psi, forms=True)
    )


def right_term(phi, psi):
    p = phi.degree_left
    q = psi.degree_left

    return sum(
        (-1) ** k
        / (math.factorial(q - k) * math.factorial(p - k))
        * mf.InnerProduct(
            mf.Trace(phi.trans, l=p - k),
            mf.Trace(psi, l=q - k),
            forms=True,
        )
        for k in range(0, min(p, q) + 1)
    )


left = mf.SlotInnerProduct(dg.Wedge(B11, C11))
mid = middle_term(B11, C11)
right = right_term(B11, C11)
print(f"L2 error doubleform identity (1,1)-(1,1): {l2_error(left, mid, mesh3):.6f}")
assert l2_error(left, mid, mesh3) < TOL
print(f"L2 error doubleform identity (1,1)-(1,1): {l2_error(mid, right, mesh3):.6f}")
assert l2_error(mid, right, mesh3) < TOL

left = mf.SlotInnerProduct(dg.Wedge(A22, B11))
mid = middle_term(A22, B11)
right = right_term(A22, B11)
print(f"L2 error doubleform identity (2,2)-(1,1): {l2_error(left, mid, mesh3):.6f}")
assert l2_error(left, mid, mesh3) < TOL
print(f"L2 error doubleform identity (2,2)-(1,1): {l2_error(mid, right, mesh3):.6f}")
assert l2_error(mid, right, mesh3) < TOL

left = mf.SlotInnerProduct(dg.Wedge(C11, B22))
mid = middle_term(C11, B22)
right = right_term(C11, B22)
print(f"L2 error doubleform identity (1,1)-(2,2): {l2_error(left, mid, mesh3):.6f}")
assert l2_error(left, mid, mesh3) < TOL
print(f"L2 error doubleform identity (1,1)-(2,2): {l2_error(mid, right, mesh3):.6f}")
assert l2_error(mid, right, mesh3) < TOL


For all $\sigma\in\Lambda^{1,1}(\Omega)$ there holds
$$
\begin{align*}
&\mathcal{S}\sigma = \frac{-1}{(N-2)!}\star^{-1}(g^{N-2}\wedge\sigma),\\
&\mathcal{J}\sigma = \frac{-1}{(N-3)!}\star^{-1}(g^{N-3}\wedge\sigma)
\end{align*}
$$

In [ ]:
N = 3

left = mf.S(B11)
right = -1 / math.factorial(N - 2) * mf.inv_star(dg.Wedge(mf.G, B11))
print(f"L2 error B operator (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}")
assert l2_error(left, right, mesh3) < TOL

left = mf.J(B11)
# TODO inv_star B11 is a (2,2) doubleform, but should be (1,1) double form?!
right = -1 / math.factorial(N - 3) * mf.inv_star(B11)
print(f"left norm: {l2_norm(left, mesh3):.6f}, right norm: {l2_norm(right, mesh3):.6f}")
# print(f"L2 error J operator (1,1)-doubleforms: {l2_error(left, right, mesh3):.6f}")
# assert l2_error(left, right, mesh3) < TOL

Define the operator $s:\Lambda^{p,1}(\Omega)\to\Lambda^{p+1,q-1}(\Omega)$ with an orthonormal basis $\{e_i\}_{i=1}^N$ and its associated dual basis \{e^i\}_{i=1}^N$ via
$$
\begin{align*}
s(\alpha\otimes\beta)=\sum_{i=1}^N(e^i\wedge\alpha)\otimes (\iota_{e_i}\beta).
\end{align*}
$$
The maps $\mathrm{Tr}$ and $s$ commute.

In [ ]:
left = mf.Trace(mf.s(A22))
right = mf.s(mf.Trace(A22))
print(
    f"L2 error trace-s operator (2,2)-doubleforms: {l2_error(left, right, mesh3):.6f}"
)
assert l2_error(left, right, mesh3) < TOL

# TODO: double form exceeds degree
# left = mf.Trace(mf.s(A33))
# right = mf.s(mf.Trace(A33))
# print(
#     f"L2 error trace-s operator (3,3)-doubleforms: {l2_error(left, right, mesh3):.6f}"
# )
# assert l2_error(left, right, mesh3) < TOL